# LeetCode 2908 — Minimum Sum of Mountain Triplets I bằng Python

Link đề: https://leetcode.com/problems/minimum-sum-of-mountain-triplets-i/ · Độ khó: **Easy** · Chủ đề: Array, Prefix/Suffix

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def minimumSum(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **bộ ba chỉ số** `(i, j, k)` với `i < j < k` tạo hình **"ngọn núi"** (`nums[i] < nums[j]` và `nums[k] < nums[j]`) sao cho **tổng ba phần tử nhỏ nhất**; không có thì trả `-1`.
2. Đi từ cách **ba vòng lặp lồng nhau `O(n³)`** → **cố định đỉnh `j`, tìm min bên trái và bên phải `O(n²)`** → **mảng `prefix_min` / `suffix_min` `O(n)`** → **một mảng `suffix_min` + biến `prefix_min` chạy `O(n)` thời gian, ít bộ nhớ hơn**.
3. Hiểu **ý tưởng "cố định phần tử giữa"**: khi đỉnh `j` đã chọn thì hai chân núi **độc lập nhau**, mỗi bên chỉ cần lấy **giá trị nhỏ nhất**.
4. Hiểu **prefix minimum / suffix minimum** (giá trị nhỏ nhất của tiền tố / hậu tố) và cách xây bằng một vòng lặp.
5. Tránh các lỗi kinh điển: dùng `<=` thay vì `<`, tính prefix min **gồm cả `j`**, lấy min toàn mảng thay vì min **một bên**, quên trả `-1`, và đặt `INF` quá nhỏ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` đánh chỉ số từ `0`. Một bộ ba `(i, j, k)` gọi là **bộ ba ngọn núi (mountain triplet)** nếu thỏa **cả ba** điều kiện:

- `i < j < k`;
- `nums[i] < nums[j]`;
- `nums[k] < nums[j]`.

Hãy trả về **tổng nhỏ nhất** `nums[i] + nums[j] + nums[k]` của một bộ ba ngọn núi. Nếu **không tồn tại** bộ ba nào, trả về `-1`.

**Ví dụ 1**

- Input: `nums = [8, 6, 1, 5, 3]`
- Output: `9`
- Giải thích: bộ ba `(2, 3, 4)` có `nums[2] = 1 < nums[3] = 5` và `nums[4] = 3 < nums[3] = 5`, tổng `1 + 5 + 3 = 9`. Có thể kiểm tra đây là tổng nhỏ nhất.

**Ví dụ 2**

- Input: `nums = [5, 4, 8, 7, 10, 2]`
- Output: `13`
- Giải thích: bộ ba `(1, 3, 5)`: `4 < 7` và `2 < 7`, tổng `4 + 7 + 2 = 13`.

**Ví dụ 3**

- Input: `nums = [6, 5, 4, 3, 4, 5]`
- Output: `-1`
- Giải thích: không có bộ ba ngọn núi nào, nên trả `-1`.

**Ràng buộc:** `3 <= nums.length <= 50`; `1 <= nums[i] <= 50`.

In [1]:
from typing import List
import random

examples = [
    ([8, 6, 1, 5, 3], 9),
    ([5, 4, 8, 7, 10, 2], 13),
    ([6, 5, 4, 3, 4, 5], -1),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [8, 6, 1, 5, 3] -> kỳ vọng 9
nums = [5, 4, 8, 7, 10, 2] -> kỳ vọng 13
nums = [6, 5, 4, 3, 4, 5] -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Chỉ số, `range` và các điều kiện `i < j < k`

Bộ ba `(i, j, k)` có `i < j < k` nghĩa là ba **vị trí khác nhau, theo thứ tự trái sang phải**. Ba vòng lặp lồng nhau có thể đảm bảo điều đó: `i` chạy `0..n-3`, `j` chạy `i+1..n-2`, `k` chạy `j+1..n-1`. Ta kiểm bằng cách liệt kê các bộ ba của một mảng ngắn.

In [2]:
n = 4
triples = [(i, j, k) for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n)]
print(triples)            # [(0,1,2), (0,1,3), (0,2,3), (1,2,3)]
print(len(triples))       # 4 = C(4, 3)

[(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]
4


### 3.2. "Ngọn núi" là gì? Hãy phân tích một bộ ba

Với `nums = [8, 6, 1, 5, 3]`:

- Bộ ba `(2, 3, 4)`: giá trị `(1, 5, 3)`: đi **lên** (1 -> 5) rồi **xuống** (5 -> 3). Đỉnh `5` ở giữa cao hơn hai đầu.
- Bộ ba `(0, 1, 2)`: giá trị `(8, 6, 1)`: chỉ **đi xuống**, không phải núi.

Tức là phần tử **giữa** phải **lớn hơn cả hai đầu**.

In [3]:
nums = [8, 6, 1, 5, 3]
def is_mountain(i, j, k):
    return i < j < k and nums[i] < nums[j] and nums[k] < nums[j]

print(is_mountain(2, 3, 4), (nums[2], nums[3], nums[4]))    # True  (1, 5, 3)
print(is_mountain(0, 1, 2), (nums[0], nums[1], nums[2]))    # False (8, 6, 1)

True (1, 5, 3)
False (8, 6, 1)


### 3.3. Hàm `min`, hằng `inf` và "chưa có đáp án"

`min(a, b)` lấy số nhỏ hơn; `min(danh_sách)` lấy nhỏ nhất của cả danh sách. Để biểu diễn "chưa tìm thấy đáp án", ta khởi tạo biến kết quả bằng một số **rất lớn** (`INF`), ví dụ `float("inf")` hoặc `10**9`. Cuối cùng nếu kết quả vẫn bằng `INF` nghĩa là không có bộ ba hợp lệ, trả `-1`. Khi chọn `INF` phải **lớn hơn mọi tổng có thể** (ở đây tổng tối đa `3 · 50 = 150`).

In [4]:
INF = float("inf")
best = INF
for value in [30, 12, 99]:
    best = min(best, value)
print(best)                   # 12
print(INF > 10**18)           # True: inf lớn hơn mọi số hữu hạn
print(min([4, 2, 7]), min(4, 2))

12
True
2 2


### 3.4. Prefix minimum: nhỏ nhất của tiền tố

`prefix_min[j]` = **giá trị nhỏ nhất của `nums[0..j-1]`** (các phần tử **đứng trước** `j`, không gồm `j`). Xây bằng một vòng lặp: `prefix_min[j] = min(prefix_min[j-1], nums[j-1])`. Với `j = 0` chưa có phần tử nào nên đặt `INF`.

In [5]:
nums = [8, 6, 1, 5, 3]
INF = float("inf")
n = len(nums)
prefix_min = [INF] * n
for j in range(1, n):
    prefix_min[j] = min(prefix_min[j - 1], nums[j - 1])
print(prefix_min)            # [inf, 8, 6, 1, 1]

[inf, 8, 6, 1, 1]


### 3.5. Suffix minimum: nhỏ nhất của hậu tố

`suffix_min[j]` = **giá trị nhỏ nhất của `nums[j+1..n-1]`** (các phần tử **đứng sau** `j`). Xây **từ phải sang trái**: `suffix_min[j] = min(suffix_min[j+1], nums[j+1])`; với `j = n - 1` đặt `INF`.

In [6]:
suffix_min = [INF] * n
for j in range(n - 2, -1, -1):
    suffix_min[j] = min(suffix_min[j + 1], nums[j + 1])
print(suffix_min)            # [1, 1, 3, 3, inf]

[1, 1, 3, 3, inf]


### 3.6. Ghép lại: tại mỗi `j` kiểm tra hai điều kiện

Tại chỉ số `j`: chân trái tốt nhất là `prefix_min[j]`, chân phải tốt nhất là `suffix_min[j]`. Núi hợp lệ nếu **cả hai nhỏ hơn `nums[j]`** (`INF` tự động không nhỏ hơn). Khi đó tổng nhỏ nhất ứng với đỉnh `j` là `prefix_min[j] + nums[j] + suffix_min[j]`.

In [7]:
for j in range(n):
    ok = prefix_min[j] < nums[j] and suffix_min[j] < nums[j]
    total = prefix_min[j] + nums[j] + suffix_min[j] if ok else None
    print(f"j={j}: nums[j]={nums[j]}, left_min={prefix_min[j]}, right_min={suffix_min[j]} -> {'tổng ' + str(total) if ok else 'không phải đỉnh'}")

j=0: nums[j]=8, left_min=inf, right_min=1 -> không phải đỉnh
j=1: nums[j]=6, left_min=8, right_min=1 -> không phải đỉnh
j=2: nums[j]=1, left_min=6, right_min=3 -> không phải đỉnh
j=3: nums[j]=5, left_min=1, right_min=3 -> tổng 9
j=4: nums[j]=3, left_min=1, right_min=inf -> không phải đỉnh


### 3.7. Duyệt hai chiều: `range(n - 2, -1, -1)`

`range(start, stop, step)` với `step = -1` đếm **ngược**: `range(n - 2, -1, -1)` sinh `n-2, n-3, ..., 0`. Dừng tại `-1` (không gồm). Dùng để xây `suffix_min` từ phải sang trái.

In [8]:
print(list(range(3, -1, -1)))          # [3, 2, 1, 0]
print(list(range(10 - 2, -1, -1))[:4]) # [8, 7, 6, 5]

[3, 2, 1, 0]
[8, 7, 6, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy nhìn mảng như một dãy **cột cao thấp**. Ta tìm một **cột giữa** (đỉnh) mà bên **trái** có cột **thấp hơn** và bên **phải** cũng có cột **thấp hơn**; rồi ta chọn cột thấp **nhất** ở mỗi bên (để tổng nhỏ nhất).

Ví dụ 1: `nums = [8, 6, 1, 5, 3]`

| đỉnh `j` | `nums[j]` | min bên trái | min bên phải | đủ điều kiện? | tổng |
|---|---|---|---|---|---|
| 1 | 6 | 8 | 1 | không (8 không nhỏ hơn 6) | - |
| 2 | 1 | 6 | 3 | không (6 không nhỏ hơn 1) | - |
| 3 | 5 | 1 | 3 | **có** (1 < 5 và 3 < 5) | `1 + 5 + 3 = 9` |

Đỉnh `0` và `4` không có phần tử ở một bên nên không thể là đỉnh. Đáp án là tổng nhỏ nhất qua mọi đỉnh hợp lệ: `9`.

## 5. Cách 1 — Brute Force / baseline: ba vòng lặp

**Ý tưởng.** Thử **mọi bộ ba** `(i, j, k)` với `i < j < k`; nếu là núi thì cập nhật tổng nhỏ nhất. Đây là mô phỏng trực tiếp đề bài.

In [9]:
def minimum_sum_brute(nums: List[int]) -> int:
    n = len(nums)
    best = float("inf")                       # chưa có đáp án
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] < nums[j] and nums[k] < nums[j]:     # núi
                    best = min(best, nums[i] + nums[j] + nums[k])
    return best if best != float("inf") else -1


for nums, expected in examples:
    print(nums, "->", minimum_sum_brute(nums), "(kỳ vọng", expected, ")")

[8, 6, 1, 5, 3] -> 9 (kỳ vọng 9 )
[5, 4, 8, 7, 10, 2] -> 13 (kỳ vọng 13 )
[6, 5, 4, 3, 4, 5] -> -1 (kỳ vọng -1 )


In [10]:
# Liệt kê các bộ ba núi của ví dụ 1 để thấy đáp án nhỏ nhất
nums = [8, 6, 1, 5, 3]
found = []
for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
        for k in range(j + 1, len(nums)):
            if nums[i] < nums[j] and nums[k] < nums[j]:
                found.append(((i, j, k), nums[i] + nums[j] + nums[k]))
print(found)
print("nhỏ nhất:", min(s for _, s in found))

[((2, 3, 4), 9)]
nhỏ nhất: 9


**Độ phức tạp (Cách 1).** Gọi `n` là độ dài `nums`. Ba vòng lồng nhau: **Time `O(n³)`**; chỉ vài biến: **Space `O(1)`**. Với `n <= 50` (khoảng 20 000 bộ ba) vẫn chạy kịp.

**Nút thắt.** Mỗi bộ ba kiểm tra riêng nên làm lại rất nhiều việc: với một đỉnh `j` cố định, ta kiểm tra **cùng một `nums[i]`** nhiều lần cho nhiều `k`. Có cách tốt hơn: chọn đỉnh rồi xử lý hai bên **độc lập**.

## 6. Ý tưởng thuật toán / bước đột phá

**Cố định đỉnh `j`.** Điều kiện hai chân núi `nums[i] < nums[j]` (với `i < j`) và `nums[k] < nums[j]` (với `k > j`) **không liên quan nhau**. Do đó, để **tổng nhỏ nhất** với đỉnh `j`:

- chọn `nums[i]` **nhỏ nhất** trong các phần tử đứng trước `j` thỏa `nums[i] < nums[j]`;
- chọn `nums[k]` **nhỏ nhất** trong các phần tử đứng sau `j` thỏa `nums[k] < nums[j]`.

Nhận xét then chốt: **phần tử nhỏ nhất của cả bên trái** thỏa điều kiện `< nums[j]` **nếu và chỉ nếu bên trái có phần tử nào** `< nums[j]`. Vậy chỉ cần dùng `prefix_min[j]` và `suffix_min[j]`: nếu `prefix_min[j] < nums[j]` thì nó chính là chân trái tốt nhất, ngược lại không có chân trái hợp lệ.

## 7. Các cách tiếp theo

### Cách 2 — Cố định đỉnh `j`, quét hai bên `O(n²)`

**Giải thích.** Với mỗi `j`, quét trái tìm `left` = min các `nums[i] < nums[j]`, quét phải tìm `right` = min các `nums[k] < nums[j]`. Nếu cả hai tồn tại thì cập nhật đáp án. Mỗi `j` tốn `O(n)`.

In [11]:
def minimum_sum_fix_middle(nums: List[int]) -> int:
    n = len(nums)
    INF = float("inf")
    best = INF
    for j in range(1, n - 1):                              # đỉnh không thể là đầu hay cuối
        left = INF
        for i in range(j):                                 # quét các phần tử bên trái
            if nums[i] < nums[j]:
                left = min(left, nums[i])
        right = INF
        for k in range(j + 1, n):                          # quét các phần tử bên phải
            if nums[k] < nums[j]:
                right = min(right, nums[k])
        if left != INF and right != INF:
            best = min(best, left + nums[j] + right)
    return best if best != INF else -1


for nums, expected in examples:
    print(nums, "->", minimum_sum_fix_middle(nums), "(kỳ vọng", expected, ")")

[8, 6, 1, 5, 3] -> 9 (kỳ vọng 9 )
[5, 4, 8, 7, 10, 2] -> 13 (kỳ vọng 13 )
[6, 5, 4, 3, 4, 5] -> -1 (kỳ vọng -1 )


**Độ phức tạp Cách 2:** **Time `O(n²)`** (mỗi `j` quét `O(n)`), **Space `O(1)`**.

### Cách 3 — Hai mảng `prefix_min` và `suffix_min` `O(n)`

**Giải thích.** Xây `prefix_min[j]` (nhỏ nhất các phần tử trước `j`) và `suffix_min[j]` (nhỏ nhất các phần tử sau `j`) trong `O(n)`. Rồi duyệt đỉnh `j`: nếu `prefix_min[j] < nums[j]` và `suffix_min[j] < nums[j]` thì cập nhật đáp án bằng tổng ba số.

In [12]:
def minimum_sum_prefix_suffix(nums: List[int]) -> int:
    n = len(nums)
    INF = float("inf")
    prefix_min = [INF] * n                       # nhỏ nhất của nums[0..j-1]
    for j in range(1, n):
        prefix_min[j] = min(prefix_min[j - 1], nums[j - 1])
    suffix_min = [INF] * n                       # nhỏ nhất của nums[j+1..n-1]
    for j in range(n - 2, -1, -1):
        suffix_min[j] = min(suffix_min[j + 1], nums[j + 1])

    best = INF
    for j in range(n):
        if prefix_min[j] < nums[j] and suffix_min[j] < nums[j]:
            best = min(best, prefix_min[j] + nums[j] + suffix_min[j])
    return best if best != INF else -1


for nums, expected in examples:
    print(nums, "->", minimum_sum_prefix_suffix(nums), "(kỳ vọng", expected, ")")

[8, 6, 1, 5, 3] -> 9 (kỳ vọng 9 )
[5, 4, 8, 7, 10, 2] -> 13 (kỳ vọng 13 )
[6, 5, 4, 3, 4, 5] -> -1 (kỳ vọng -1 )


In [13]:
# Code nộp LeetCode cho Cách 3
from typing import List

class Solution:
    def minimumSum(self, nums: List[int]) -> int:
        n = len(nums)
        INF = float("inf")
        prefix_min = [INF] * n
        for j in range(1, n):
            prefix_min[j] = min(prefix_min[j - 1], nums[j - 1])
        suffix_min = [INF] * n
        for j in range(n - 2, -1, -1):
            suffix_min[j] = min(suffix_min[j + 1], nums[j + 1])
        best = INF
        for j in range(n):
            if prefix_min[j] < nums[j] and suffix_min[j] < nums[j]:
                best = min(best, prefix_min[j] + nums[j] + suffix_min[j])
        return best if best != INF else -1

**Độ phức tạp Cách 3:** **Time `O(n)`** (ba lần duyệt tuyến tính), **Space `O(n)`** (hai mảng phụ).

### Cách 4 — Một mảng `suffix_min` và biến `prefix_min` chạy

**Giải thích.** Chỉ cần lưu mảng `suffix_min` (xây từ phải); phần `prefix_min` thì giữ **một biến** cập nhật khi duyệt `j` từ trái sang phải: dùng `left_min` hiện tại cho `j`, rồi sau khi xử lý `j` mới cập nhật `left_min = min(left_min, nums[j])`. Tiết kiệm một mảng.

In [14]:
def minimum_sum_one_array(nums: List[int]) -> int:
    n = len(nums)
    INF = float("inf")
    suffix_min = [INF] * n
    for j in range(n - 2, -1, -1):
        suffix_min[j] = min(suffix_min[j + 1], nums[j + 1])

    best = INF
    left_min = INF                                   # nhỏ nhất của nums[0..j-1]
    for j in range(n):
        if left_min < nums[j] and suffix_min[j] < nums[j]:
            best = min(best, left_min + nums[j] + suffix_min[j])
        left_min = min(left_min, nums[j])            # cập nhật SAU khi dùng cho j
    return best if best != INF else -1


for nums, expected in examples:
    print(nums, "->", minimum_sum_one_array(nums), "(kỳ vọng", expected, ")")

[8, 6, 1, 5, 3] -> 9 (kỳ vọng 9 )
[5, 4, 8, 7, 10, 2] -> 13 (kỳ vọng 13 )
[6, 5, 4, 3, 4, 5] -> -1 (kỳ vọng -1 )


In [15]:
# Code nộp LeetCode cho Cách 4
from typing import List

class Solution:
    def minimumSum(self, nums: List[int]) -> int:
        n = len(nums)
        INF = float("inf")
        suffix_min = [INF] * n
        for j in range(n - 2, -1, -1):
            suffix_min[j] = min(suffix_min[j + 1], nums[j + 1])
        best, left_min = INF, INF
        for j in range(n):
            if left_min < nums[j] and suffix_min[j] < nums[j]:
                best = min(best, left_min + nums[j] + suffix_min[j])
            left_min = min(left_min, nums[j])
        return best if best != INF else -1

**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (một mảng `suffix_min`).

> Cách 3 và Cách 4 đều `O(n)`; Cách 3 rõ ràng hơn để giảng, Cách 4 ngắn hơn về bộ nhớ. Với `n <= 50` mọi cách đều nhanh; Cách 1 hữu ích để đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 3 trên `nums = [5, 4, 8, 7, 10, 2]` (ví dụ 2): hai mảng min rồi từng đỉnh.

In [16]:
def trace_prefix_suffix(nums):
    n, INF = len(nums), float("inf")
    prefix_min, suffix_min = [INF] * n, [INF] * n
    for j in range(1, n):
        prefix_min[j] = min(prefix_min[j - 1], nums[j - 1])
    for j in range(n - 2, -1, -1):
        suffix_min[j] = min(suffix_min[j + 1], nums[j + 1])
    print("nums       =", nums)
    print("prefix_min =", prefix_min)
    print("suffix_min =", suffix_min)
    best = INF
    for j in range(n):
        ok = prefix_min[j] < nums[j] and suffix_min[j] < nums[j]
        if ok:
            total = prefix_min[j] + nums[j] + suffix_min[j]
            best = min(best, total)
            print(f"j={j}: nums[j]={nums[j]:>2} left={prefix_min[j]} right={suffix_min[j]} -> núi, tổng {total}, best={best}")
        else:
            print(f"j={j}: nums[j]={nums[j]:>2} left={prefix_min[j]} right={suffix_min[j]} -> không")
    print("=>", best if best != INF else -1)
    return best if best != INF else -1


assert trace_prefix_suffix([5, 4, 8, 7, 10, 2]) == 13

nums       = [5, 4, 8, 7, 10, 2]
prefix_min = [inf, 5, 4, 4, 4, 4]
suffix_min = [2, 2, 2, 2, 2, inf]
j=0: nums[j]= 5 left=inf right=2 -> không
j=1: nums[j]= 4 left=5 right=2 -> không
j=2: nums[j]= 8 left=4 right=2 -> núi, tổng 14, best=14
j=3: nums[j]= 7 left=4 right=2 -> núi, tổng 13, best=13
j=4: nums[j]=10 left=4 right=2 -> núi, tổng 16, best=13
j=5: nums[j]= 2 left=4 right=inf -> không
=> 13


In [17]:
def trace_brute(nums):
    n, best = len(nums), float("inf")
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                count += 1
                if nums[i] < nums[j] and nums[k] < nums[j]:
                    s = nums[i] + nums[j] + nums[k]
                    best = min(best, s)
                    print(f"(i,j,k)=({i},{j},{k}) giá trị ({nums[i]},{nums[j]},{nums[k]}) -> núi, tổng {s}, best={best}")
    print("đã xét", count, "bộ ba")
    return best if best != float("inf") else -1


assert trace_brute([8, 6, 1, 5, 3]) == 9

(i,j,k)=(2,3,4) giá trị (1,5,3) -> núi, tổng 9, best=9
đã xét 10 bộ ba


Bảng tóm tắt (Cách 3, `nums = [5, 4, 8, 7, 10, 2]`):

| `j` | `nums[j]` | `prefix_min[j]` | `suffix_min[j]` | núi? | tổng |
|---|---|---|---|---|---|
| 0 | 5 | inf | 2 | không | |
| 1 | 4 | 5 | 2 | không (5 không nhỏ hơn 4) | |
| 2 | 8 | 4 | 2 | có | 14 |
| 3 | 7 | 4 | 2 | có | **13** |
| 4 | 10 | 4 | 2 | có | 16 |
| 5 | 2 | 4 | inf | không | |

Đáp án `min(14, 13, 16) = 13`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (prefix/suffix). Đúng class `Solution`, đúng method `minimumSum(self, nums: List[int]) -> int`.

In [18]:
from typing import List

class Solution:
    def minimumSum(self, nums: List[int]) -> int:
        n = len(nums)
        INF = float("inf")
        prefix_min = [INF] * n                   # nhỏ nhất của nums[0..j-1]
        for j in range(1, n):
            prefix_min[j] = min(prefix_min[j - 1], nums[j - 1])
        suffix_min = [INF] * n                   # nhỏ nhất của nums[j+1..n-1]
        for j in range(n - 2, -1, -1):
            suffix_min[j] = min(suffix_min[j + 1], nums[j + 1])

        best = INF
        for j in range(n):
            if prefix_min[j] < nums[j] and suffix_min[j] < nums[j]:
                best = min(best, prefix_min[j] + nums[j] + suffix_min[j])
        return best if best != INF else -1


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.minimumSum(nums))

[8, 6, 1, 5, 3] -> 9
[5, 4, 8, 7, 10, 2] -> 13
[6, 5, 4, 3, 4, 5] -> -1


## 10. Vì sao thuật toán đúng

- **Mọi bộ ba núi có một đỉnh `j`:** với `i < j < k`, đỉnh là chỉ số giữa. Ta chia bài toán theo đỉnh và lấy **min qua mọi `j`**; nên đáp án là tổng nhỏ nhất theo từng đỉnh.
- **Hai chân độc lập:** khi `j` cố định, điều kiện của `i` (`i < j`, `nums[i] < nums[j]`) và của `k` (`k > j`, `nums[k] < nums[j]`) không ràng buộc lẫn nhau. Do đó tổng nhỏ nhất theo `j` là `nums[j] + (min hợp lệ bên trái) + (min hợp lệ bên phải)`.
- **Min bên trái:** tập các `nums[i]` hợp lệ là tập con của `nums[0..j-1]` gồm các giá trị `< nums[j]`. Nếu tập đó khác rỗng thì phần tử nhỏ nhất **của cả tiền tố** cũng `< nums[j]` (vì nhỏ hơn hoặc bằng mọi phần tử, trong đó có phần tử hợp lệ), nên nó thuộc tập và là min của tập. Nếu `prefix_min[j] >= nums[j]` thì mọi phần tử bên trái `>= nums[j]` và tập rỗng. Tương tự cho bên phải.
- **Sentinel:** `INF` biểu thị "không có phần tử"; `INF < nums[j]` là `False` nên đỉnh đó bị loại, đúng ý.
- **Trả `-1`:** nếu không có đỉnh nào đủ điều kiện thì `best` vẫn là `INF`.
- **Tính chất kiểm được:** đáp án khác `-1` luôn bằng tổng của một bộ ba núi có thật, và không có bộ ba núi nào cho tổng nhỏ hơn (đối chiếu với brute force).

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài `nums`.

- **Cách 1:** Time `O(n³)`, Space `O(1)`.
- **Cách 2:** Time `O(n²)`, Space `O(1)`.
- **Cách 3:** Time **`O(n)`**, Space `O(n)` (hai mảng `prefix_min`, `suffix_min`).
- **Cách 4:** Time **`O(n)`**, Space `O(n)` (một mảng, ít hơn một nửa Cách 3 về hằng số).
- Có thể giảm Space xuống `O(1)` đáng kể hơn bằng cách xử lý hai chiều khác, nhưng không cần với `n <= 50`.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` rồi kiểm tra bằng `assert`.

In [19]:
sol = Solution()
impls = {
    "brute": minimum_sum_brute,
    "fix_middle": minimum_sum_fix_middle,
    "prefix_suffix": minimum_sum_prefix_suffix,
    "one_array": minimum_sum_one_array,
    "submit": sol.minimumSum,
}

for nums, expected in examples:
    for key, fn in impls.items():
        assert fn(list(nums)) == expected, (key, nums, fn(list(nums)), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [20]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, nums, got, expected)
    print("PASS", label, "->", expected)


check_all([1, 3, 2], 6, "n = 3, đúng một bộ ba núi")
check_all([1, 2, 3], -1, "tăng dần: không có chân phải")
check_all([3, 2, 1], -1, "giảm dần: không có chân trái")
check_all([5, 5, 5], -1, "bằng nhau: điều kiện phải là nhỏ hơn hẳn")
check_all([1, 5, 1], 7, "đối xứng, hai chân bằng nhau")
check_all([2, 1, 3, 1, 2], 1 + 3 + 1, "đỉnh 3 với chân trái 1 và chân phải 1")
check_all([4, 4, 4, 9, 1, 4], 4 + 9 + 1, "chân trái 4 (min trái), chân phải 1")
check_all([50, 1, 50, 1, 50], 1 + 50 + 1, "giá trị lớn nhất 50")

PASS n = 3, đúng một bộ ba núi -> 6
PASS tăng dần: không có chân phải -> -1
PASS giảm dần: không có chân trái -> -1
PASS bằng nhau: điều kiện phải là nhỏ hơn hẳn -> -1
PASS đối xứng, hai chân bằng nhau -> 7
PASS đỉnh 3 với chân trái 1 và chân phải 1 -> 5
PASS chân trái 4 (min trái), chân phải 1 -> 14
PASS giá trị lớn nhất 50 -> 52


### 12.2. Tính chất và phản ví dụ

In [21]:
# Tính chất 1: đáp án khác -1 luôn >= 1 + 2 + 1 = 4? (đỉnh phải >= 2, hai chân >= 1)
random.seed(8)
for _ in range(300):
    n = random.randint(3, 12)
    nums = [random.randint(1, 50) for _ in range(n)]
    r = sol.minimumSum(nums)
    assert r == -1 or r >= 4
print("PASS đáp án là -1 hoặc >= 4 (đỉnh >= 2, mỗi chân >= 1)")

# Tính chất 2: đáp án khác -1 là tổng của một bộ ba núi thật
def exists_mountain_with_sum(nums, s):
    n = len(nums)
    return any(nums[i] < nums[j] > nums[k] and nums[i] + nums[j] + nums[k] == s
               for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n))

for _ in range(300):
    n = random.randint(3, 12)
    nums = [random.randint(1, 50) for _ in range(n)]
    r = sol.minimumSum(nums)
    if r != -1:
        assert exists_mountain_with_sum(nums, r)
print("PASS đáp án là tổng của một bộ ba núi có thật")

# Đầu vào không bị sửa
nums = [5, 4, 8, 7, 10, 2]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: dùng <= thay vì < (cho phép bằng nhau)
def sai_bang(nums):
    n, best = len(nums), float("inf")
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] <= nums[j] and nums[k] <= nums[j]:
                    best = min(best, nums[i] + nums[j] + nums[k])
    return best if best != float("inf") else -1

assert sai_bang([5, 5, 5]) == 15 and sol.minimumSum([5, 5, 5]) == -1
print("PASS phản ví dụ: <= cho", sai_bang([5, 5, 5]), "thay vì -1")

# Phản ví dụ 2: lấy min của CẢ MẢNG thay vì min một bên
def sai_min_toan_mang(nums):
    m = min(nums)
    cands = [m + nums[j] + m for j in range(1, len(nums) - 1) if m < nums[j]]
    return min(cands) if cands else -1

assert sai_min_toan_mang([8, 6, 1, 5, 3]) != 9
print("PASS phản ví dụ: min toàn mảng cho", sai_min_toan_mang([8, 6, 1, 5, 3]), "thay vì 9")

PASS đáp án là -1 hoặc >= 4 (đỉnh >= 2, mỗi chân >= 1)
PASS đáp án là tổng của một bộ ba núi có thật
PASS đầu vào không bị sửa
PASS phản ví dụ: <= cho 15 thay vì -1
PASS phản ví dụ: min toàn mảng cho 7 thay vì 9


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [22]:
random.seed(2908)
for trial in range(3000):
    n = random.randint(3, 50)
    hi = random.choice([3, 10, 50])
    nums = [random.randint(1, hi) for _ in range(n)]
    ref = minimum_sum_brute(list(nums))
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums, fn(list(nums)), ref)
print("PASS 3000 mảng ngẫu nhiên (n tới 50) - mọi cách khớp nhau")

PASS 3000 mảng ngẫu nhiên (n tới 50) - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` thay vì `<`

In [23]:
def sai_dung_bang(nums):
    n, best = len(nums), float("inf")
    for j in range(1, n - 1):
        left = min(nums[:j])
        right = min(nums[j + 1:])
        if left <= nums[j] and right <= nums[j]:           # sai: cho phép bằng
            best = min(best, left + nums[j] + right)
    return best if best != float("inf") else -1


print("sai  :", sai_dung_bang([5, 5, 5]), "| đúng:", Solution().minimumSum([5, 5, 5]))
print("sai  :", sai_dung_bang([3, 3, 1]), "| đúng:", Solution().minimumSum([3, 3, 1]))

sai  : 15 | đúng: -1
sai  : 7 | đúng: -1


**Vì sao sai:** đề yêu cầu `nums[i] < nums[j]` và `nums[k] < nums[j]` (**nhỏ hơn hẳn**). Dùng `<=` chấp nhận hai phần tử bằng nhau, tạo "núi" không tồn tại và trả tổng thay vì `-1`.

### Lỗi 2: Quên kiểm tra điều kiện "núi" trước khi cộng

In [24]:
def sai_quen_kiem_tra(nums):
    n, best = len(nums), float("inf")
    for j in range(1, n - 1):
        left, right = min(nums[:j]), min(nums[j + 1:])
        best = min(best, left + nums[j] + right)          # cộng luôn, không kiểm tra left < nums[j] và right < nums[j]
    return best if best != float("inf") else -1


for nums in ([1, 2, 3], [3, 2, 1], [6, 5, 4, 3, 4, 5]):
    print(nums, "sai:", sai_quen_kiem_tra(nums), "| đúng:", Solution().minimumSum(nums))

[1, 2, 3] sai: 6 | đúng: -1
[3, 2, 1] sai: 6 | đúng: -1
[6, 5, 4, 3, 4, 5] sai: 11 | đúng: -1


**Vì sao sai:** lấy min hai bên chưa đủ: phải **kiểm tra** `left < nums[j]` và `right < nums[j]`. Nếu quên, mảng tăng dần `[1, 2, 3]` bị coi là có núi (tổng `1 + 2 + 3 = 6`) trong khi đáp án đúng là `-1`; ví dụ 3 cũng bị trả một tổng thay vì `-1`.

### Lỗi 3: Lấy min của cả mảng (không phân biệt bên trái/phải)

In [25]:
def sai_toan_mang(nums):
    m = min(nums)
    cands = [m + nums[j] + m for j in range(1, len(nums) - 1) if m < nums[j]]
    return min(cands) if cands else -1


print("sai  :", sai_toan_mang([8, 6, 1, 5, 3]), "| đúng:", Solution().minimumSum([8, 6, 1, 5, 3]))
print("sai  :", sai_toan_mang([6, 5, 4, 3, 4, 5]), "| đúng:", Solution().minimumSum([6, 5, 4, 3, 4, 5]))

sai  : 7 | đúng: 9
sai  : 10 | đúng: -1


**Vì sao sai:** hai chân phải nằm **hai phía khác nhau** của đỉnh (`i < j < k`) và là **hai phần tử khác nhau**. Dùng min của cả mảng hai lần có thể dùng cùng một phần tử cho cả hai chân, hoặc lấy phần tử ở sai phía.

### Lỗi 4: Quên trả `-1` hoặc trả `inf`

In [26]:
def sai_quen_tra_ve(nums):
    n, INF = len(nums), float("inf")
    best = INF
    for j in range(1, n - 1):
        left, right = min(nums[:j]), min(nums[j + 1:])
        if left < nums[j] and right < nums[j]:
            best = min(best, left + nums[j] + right)
    return best                                   # thiếu: nếu best == INF thì trả -1


print("sai  :", sai_quen_tra_ve([1, 2, 3]), "| đúng:", Solution().minimumSum([1, 2, 3]))

sai  : inf | đúng: -1


**Vì sao sai:** khi không có núi nào, `best` vẫn là `INF` (`inf`, kiểu `float`). Đề yêu cầu trả `-1`. Luôn kiểm tra `best == INF` ở cuối.

### Lỗi 5: Duyệt đỉnh `j` ở hai đầu mảng gây `IndexError`/min rỗng

In [27]:
try:
    nums = [3, 1, 2]
    j = 0
    print(min(nums[:j]))                 # nums[:0] là danh sách rỗng
except ValueError as e:
    print("ValueError:", e)
print("đúng: chỉ xét j từ 1 đến n-2 ->", list(range(1, len(nums) - 1)))

ValueError: min() iterable argument is empty
đúng: chỉ xét j từ 1 đến n-2 -> [1]


**Vì sao sai:** đỉnh `j = 0` không có phần tử bên trái (`nums[:0]` rỗng) và `j = n - 1` không có bên phải; `min` của danh sách rỗng báo `ValueError`. Hoặc dùng sentinel `INF` (như Cách 3), hoặc chỉ duyệt `j` từ `1` đến `n - 2`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "bộ ba có phần tử giữa lớn nhất/nhỏ nhất", "i < j < k", "tổng nhỏ nhất" → **cố định phần tử giữa `j`**, xử lý trái/phải độc lập.
- Dùng **prefix min / suffix min** cho "min của mọi phần tử bên trái / phải".
- Công thức nhớ: núi hợp lệ tại `j` ⇔ `prefix_min[j] < nums[j]` **và** `suffix_min[j] < nums[j]`; tổng `= prefix_min[j] + nums[j] + suffix_min[j]`.
- Với `n` nhỏ (như ở bài này) cứ viết brute force `O(n³)` trước để đối chiếu.
- Sentinel `INF` cho "chưa có"; nhớ đổi `INF` thành `-1` ở cuối.
- Cùng kỹ thuật: Trapping Rain Water (prefix max/suffix max), Product of Array Except Self.

## 15. Biến thể của bài

1. **Bài II** (cùng series): `n` lên tới `10^5` và `nums[i]` tới `10^8`: bắt buộc `O(n)`, brute force `O(n³)` không đủ.
2. Tìm **tổng lớn nhất** của bộ ba núi (dùng prefix max, nhưng chú ý ràng buộc `< nums[j]` nên lấy **lớn nhất** các giá trị `< nums[j]`, cần cấu trúc khác như sorted list).
3. Bộ ba **thung lũng** (`nums[i] > nums[j]` và `nums[k] > nums[j]`) có tổng nhỏ nhất.
4. **Đếm** số bộ ba núi (không cần tổng).
5. Với mỗi `j` in ra **bộ ba tối ưu** `(i, j, k)` thay vì chỉ tổng.
6. Cho phép **bằng nhau** (`<=`) và so sánh khác biệt.

Ta thử biến thể 3 (thung lũng) và 4 (đếm):

In [28]:
def minimum_sum_valley(nums):
    """Biến thể 3: nums[i] > nums[j] < nums[k], tổng nhỏ nhất (O(n^2): lấy min các giá trị LỚN HƠN nums[j] mỗi bên)."""
    n, best = len(nums), float("inf")
    for j in range(1, n - 1):
        lefts = [x for x in nums[:j] if x > nums[j]]
        rights = [x for x in nums[j + 1:] if x > nums[j]]
        if lefts and rights:
            best = min(best, min(lefts) + nums[j] + min(rights))
    return best if best != float("inf") else -1


def count_mountains(nums):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n)
               if nums[i] < nums[j] > nums[k])


print("3. thung lũng nhỏ nhất của [5, 1, 6, 9, 2, 8]:", minimum_sum_valley([5, 1, 6, 9, 2, 8]))
print("4. số bộ ba núi của [8, 6, 1, 5, 3]:", count_mountains([8, 6, 1, 5, 3]))

3. thung lũng nhỏ nhất của [5, 1, 6, 9, 2, 8]: 8
4. số bộ ba núi của [8, 6, 1, 5, 3]: 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng lặp | thử mọi `(i, j, k)` | `O(n³)` | `O(1)` | `n` nhỏ; bản đối chiếu |
| 2. Cố định đỉnh, quét hai bên | min trái, min phải cho từng `j` | `O(n²)` | `O(1)` | `n` vài nghìn |
| 3. `prefix_min` + `suffix_min` | min tiền tố/hậu tố tính trước | `O(n)` | `O(n)` | **cách nộp bài**, nên dùng cho Bài II |
| 4. `suffix_min` + biến `left_min` | tiết kiệm một mảng | `O(n)` | `O(n)` | ngắn gọn, ít bộ nhớ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Solution().minimumSum([1, 3, 2])`, `Solution().minimumSum([1, 2, 3])`, `min([4, 2, 9])`, `list(range(4, -1, -1))`, `Solution().minimumSum([2, 4, 1, 4, 2])` (gợi ý: thử đỉnh `4` đầu tiên).

**Bài 2 (tính tay).** Với `nums = [9, 3, 7, 2, 8, 1]`: lập bảng `prefix_min`, `suffix_min` (dùng `INF` cho ô rỗng), rồi tìm đỉnh hợp lệ và tổng nhỏ nhất.

**Bài 3 (điền TODO).** Hoàn thành `minimum_sum_todo(nums)` theo Cách 3.

In [29]:
def minimum_sum_todo(nums):
    n = len(nums)
    INF = float("inf")
    prefix_min = [INF] * n
    suffix_min = [INF] * n
    # TODO 1: với j từ 1 đến n - 1: prefix_min[j] = min(prefix_min[j - 1], nums[j - 1])
    # TODO 2: với j từ n - 2 xuống 0: suffix_min[j] = min(suffix_min[j + 1], nums[j + 1])
    # TODO 3: với mỗi j, nếu cả hai min < nums[j] thì cập nhật best = min(best, tổng ba số)
    # TODO 4: trả về best, hoặc -1 nếu best == INF
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `minimum_sum_brute2(nums)` bằng ba vòng lặp (Cách 1) mà không nhìn code mẫu. Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `best_triplet(nums)` trả về **bộ ba chỉ số** `(i, j, k)` cho tổng nhỏ nhất (hoặc `None`). (b) Viết `maximum_sum_mountain(nums)` trả về **tổng lớn nhất** của một bộ ba núi (hoặc `-1`), có thể dùng `O(n²)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `6`, `-1`, `2`, `[4, 3, 2, 1, 0]`, `7` (đỉnh `4` đầu tiên: chân trái `2`, chân phải min `1`, tổng `2 + 4 + 1 = 7`). Bài 2: `prefix_min[j]` là min các phần tử trước `j`.
- **Mức 2 (vừa):** Bài 2: `prefix_min = [inf, 9, 3, 3, 2, 2]`, `suffix_min = [1, 1, 1, 1, 1, inf]`. Đỉnh hợp lệ: `j = 2 (7)`: `3 < 7` và `1 < 7` -> `3 + 7 + 1 = 11`; `j = 4 (8)`: `2 < 8` và `1 < 8` -> `2 + 8 + 1 = 11`; `j = 1 (3)`: `9` không nhỏ hơn 3 -> loại. Đáp án `11`. Bài 3: bốn TODO đã ghi trong comment.
- **Mức 3 (sâu):** Bài 5(a): giống Cách 4 nhưng lưu thêm chỉ số của min bên trái (`left_idx`) và của min bên phải (`suffix_idx`). Bài 5(b): với mỗi `j`, `max` các `nums[i] < nums[j]` bên trái và bên phải (chạy `O(n²)`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert Solution().minimumSum([1, 3, 2]) == 6
assert Solution().minimumSum([1, 2, 3]) == -1
assert min([4, 2, 9]) == 2
assert list(range(4, -1, -1)) == [4, 3, 2, 1, 0]
assert Solution().minimumSum([2, 4, 1, 4, 2]) == 7

# Bài 2
nums2 = [9, 3, 7, 2, 8, 1]
INF = float("inf")
pm = [INF] * 6
for j in range(1, 6):
    pm[j] = min(pm[j - 1], nums2[j - 1])
sm = [INF] * 6
for j in range(4, -1, -1):
    sm[j] = min(sm[j + 1], nums2[j + 1])
assert pm == [INF, 9, 3, 3, 2, 2] and sm == [1, 1, 1, 1, 1, INF]
assert Solution().minimumSum(nums2) == 11

# Bài 3
def minimum_sum_todo_ans(nums):
    n = len(nums)
    INF = float("inf")
    prefix_min = [INF] * n
    suffix_min = [INF] * n
    for j in range(1, n):
        prefix_min[j] = min(prefix_min[j - 1], nums[j - 1])
    for j in range(n - 2, -1, -1):
        suffix_min[j] = min(suffix_min[j + 1], nums[j + 1])
    best = INF
    for j in range(n):
        if prefix_min[j] < nums[j] and suffix_min[j] < nums[j]:
            best = min(best, prefix_min[j] + nums[j] + suffix_min[j])
    return best if best != INF else -1

for nums, expected in examples:
    assert minimum_sum_todo_ans(nums) == expected

# Bài 4
def minimum_sum_brute2(nums):
    n = len(nums)
    best = float("inf")
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] < nums[j] and nums[k] < nums[j]:
                    best = min(best, nums[i] + nums[j] + nums[k])
    return best if best != float("inf") else -1

for nums, expected in examples:
    assert minimum_sum_brute2(nums) == expected

# Bài 5(a)
def best_triplet(nums):
    n = len(nums)
    best, ans = float("inf"), None
    for j in range(1, n - 1):
        li = min(range(j), key=lambda i: nums[i])
        ri = min(range(j + 1, n), key=lambda k: nums[k])
        if nums[li] < nums[j] and nums[ri] < nums[j]:
            s = nums[li] + nums[j] + nums[ri]
            if s < best:
                best, ans = s, (li, j, ri)
    return ans

assert best_triplet([8, 6, 1, 5, 3]) == (2, 3, 4)
assert best_triplet([1, 2, 3]) is None

# Bài 5(b)
def maximum_sum_mountain(nums):
    n, best = len(nums), -1
    for j in range(1, n - 1):
        lefts = [x for x in nums[:j] if x < nums[j]]
        rights = [x for x in nums[j + 1:] if x < nums[j]]
        if lefts and rights:
            best = max(best, max(lefts) + nums[j] + max(rights))
    return best

assert maximum_sum_mountain([8, 6, 1, 5, 3]) == 1 + 5 + 3
assert maximum_sum_mountain([5, 4, 8, 7, 10, 2]) == 20      # ví dụ: 5 + 8 + 7 hoặc 8 + 10 + 2
assert maximum_sum_mountain([1, 2, 3]) == -1
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cố định đỉnh, `prefix_min` và `suffix_min`)

```python
from typing import List

class Solution:
    def minimumSum(self, nums: List[int]) -> int:
        n, INF = len(nums), float("inf")
        prefix_min, suffix_min = [INF] * n, [INF] * n
        for j in range(1, n):
            prefix_min[j] = min(prefix_min[j - 1], nums[j - 1])
        for j in range(n - 2, -1, -1):
            suffix_min[j] = min(suffix_min[j + 1], nums[j + 1])
        best = INF
        for j in range(n):
            if prefix_min[j] < nums[j] and suffix_min[j] < nums[j]:
                best = min(best, prefix_min[j] + nums[j] + suffix_min[j])
        return best if best != INF else -1
```

**Checklist**

- [ ] Điều kiện là **nhỏ hơn hẳn** (`<`) ở cả hai chân?
- [ ] `prefix_min[j]` / `suffix_min[j]` **không gồm** `nums[j]`?
- [ ] Hai chân ở **hai phía** của đỉnh (`i < j < k`)?
- [ ] Trả `-1` khi không có núi (`best == INF`)?
- [ ] `INF` lớn hơn mọi tổng có thể (ở đây tối đa 150)?
- [ ] Đã thử: `n = 3`, mảng tăng dần, giảm dần, toàn bằng nhau?
- [ ] Skeleton đúng `class Solution` / `def minimumSum(self, nums: List[int]) -> int`?